# 05 — Portfolio Composition + Diversification

**Was Du hier siehst:** wie Dein Portfolio aufgebaut ist und wo Konzentrations- bzw. Korrelations-Risiken sitzen. Keine neuen Daten — pure Diagnostik auf den bestehenden Tabellen (`pos_holdings`, `ref_instruments`, `ref_fundamentals_latest`, `mkt_quotes_daily`).

**Sektionen:**
1. Konzentration: HHI, Top-N-Weight, Effective-N
2. Allokation: nach Sektor, Industrie, Country, Currency, Asset-Type
3. Korrelations-Matrix: 90d Daily-Returns
4. Risk-Decomposition: wer trägt wieviel Portfolio-Vola (Marginal Contribution to Risk)
5. Diagnostics: Auto-erkannte Konzentrations- und Korrelations-Warnungen

**Annahmen:**
- Bewertung in `PORTFOLIO_CCY` (default EUR). FX über `mkt_fx_daily`.
- Korrelations- und Vola-Berechnung erfolgt auf *lokalen* Daily-Returns (keine FX-Adjustierung — vereinfacht; Currency-Exposure ist in den Allokations-Plots separat sichtbar).
- ETFs werden als Single-Position behandelt — die innewohnende Diversifikation eines VWRL wird NICHT aufgelöst.
- 90d ist ein bewusst kurzes Fenster für die Reaktivität; bei stabilen Schätzungen 180+ Tage nutzen.

**Was das Notebook NICHT macht:** Rebalancing-Vorschläge. Es zeigt Dir nur was *ist* — die Entscheidung was Du daraus machst (trimmen, hedgen, ignorieren) bleibt Dir.


In [ ]:
import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
import numpy as np
from datetime import date, timedelta

from modules.analysis._db import session, df
from modules.analysis._plot import setup, COLORS, hline, plt
setup()

In [ ]:
PORTFOLIO_CCY     = 'EUR'    # Bewertungs-Währung
RETURN_WINDOW_DAYS = 90      # für Korrelation + Vola
MIN_OBS_FOR_CORR   = 30      # weniger Datenpunkte -> Korrelation nicht zeigen
CONCENTRATION_WARN_PCT = 25  # Position > 25% -> Alarm
SECTOR_WARN_PCT        = 50  # Sektor > 50% -> Alarm
CORR_WARN              = 0.80  # Median pairwise corr > 0.80 -> Alarm

## 1. Holdings laden + in Portfolio-CCY bewerten

In [ ]:
with session() as con:
    holdings = df(con, '''
        SELECT h.ref_instrument_id,
               SUM(h.quantity) AS quantity,
               AVG(h.cost_per_share) AS avg_cost,
               i.symbol, i.name, i.currency AS instr_currency,
               i.asset_type, i.exchange
        FROM pos_holdings h
        LEFT JOIN ref_instruments i USING (ref_instrument_id)
        GROUP BY h.ref_instrument_id, i.symbol, i.name, i.currency, i.asset_type, i.exchange
    ''')
    # Fundamentals für sector/industry/country (optional, missing OK)
    try:
        meta = df(con, '''
            SELECT ref_instrument_id, sector, industry, country, market_cap
            FROM ref_fundamentals_latest
        ''')
    except Exception:
        meta = pd.DataFrame(columns=['ref_instrument_id', 'sector', 'industry', 'country', 'market_cap'])

    # Latest spot pro instrument
    spots = df(con, '''
        WITH ranked AS (
            SELECT ref_instrument_id, ts, close, source,
                   ROW_NUMBER() OVER (
                       PARTITION BY ref_instrument_id
                       ORDER BY ts DESC,
                                CASE source WHEN 'ib' THEN 1 WHEN 'yfinance' THEN 2 ELSE 9 END
                   ) AS rk
            FROM mkt_quotes_daily
        )
        SELECT ref_instrument_id, close AS spot FROM ranked WHERE rk = 1
    ''')

    # FX latest pro currency
    fx = df(con, '''
        WITH ranked AS (
            SELECT currency_from, currency_to, ts, rate,
                   ROW_NUMBER() OVER (PARTITION BY currency_from, currency_to ORDER BY ts DESC) AS rk
            FROM mkt_fx_daily
        )
        SELECT currency_from, currency_to, rate FROM ranked WHERE rk = 1
    ''')

if holdings.empty:
    print('Portfolio leer — Notebook übersprungen.')
else:
    pos = holdings.merge(meta, on='ref_instrument_id', how='left')
    pos = pos.merge(spots, on='ref_instrument_id', how='left')

    def fx_rate(from_ccy: str) -> float:
        if from_ccy == PORTFOLIO_CCY:
            return 1.0
        m = fx[(fx['currency_from'] == from_ccy) & (fx['currency_to'] == PORTFOLIO_CCY)]
        return float(m.iloc[0]['rate']) if not m.empty else 1.0

    pos['mv_local'] = pos['quantity'] * pos['spot']
    pos['fx_rate']  = pos['instr_currency'].apply(fx_rate)
    pos['mv_ptf']   = pos['mv_local'] * pos['fx_rate']
    total_mv = pos['mv_ptf'].sum(skipna=True)
    pos['weight'] = pos['mv_ptf'] / total_mv if total_mv > 0 else np.nan
    print(f'Total Market Value: {total_mv:,.2f} {PORTFOLIO_CCY}')
    pos[['symbol', 'instr_currency', 'quantity', 'spot', 'mv_ptf', 'weight',
          'sector', 'industry']].sort_values('weight', ascending=False).head(15)

## 2. Konzentration: HHI, Top-N, Effective-N

- **HHI** (Herfindahl-Hirschman-Index) = Σ(weight²). Range: 1/N (perfekt gleichverteilt) bis 1 (alles in einer Position). Marktübliche Schwellen aus Antitrust: <0.15 = unkonzentriert, 0.15-0.25 = moderat, >0.25 = hoch konzentriert.
- **Effective-N** = 1/HHI. "Wieviele äquivalente gleichgewichtete Positions hast Du *wirklich*?".
- **Top-3/5/10-Weight** in Prozent vom Portfolio.

In [ ]:
if not holdings.empty and total_mv > 0:
    weights_sorted = pos['weight'].dropna().sort_values(ascending=False)
    n_pos = len(weights_sorted)
    hhi = (weights_sorted ** 2).sum()
    effective_n = 1 / hhi if hhi > 0 else np.nan
    top3 = weights_sorted.head(3).sum() * 100
    top5 = weights_sorted.head(5).sum() * 100
    top10 = weights_sorted.head(10).sum() * 100
    largest = weights_sorted.iloc[0] * 100 if n_pos > 0 else 0

    print(f'  Anzahl Positions   : {n_pos}')
    print(f'  HHI                : {hhi:.3f}')
    print(f'  Effective-N        : {effective_n:.1f}  (perfekt gleichverteilt wäre {n_pos})')
    print(f'  Größte Position    : {largest:.1f}%')
    print(f'  Top-3 Weight       : {top3:.1f}%')
    print(f'  Top-5 Weight       : {top5:.1f}%')
    print(f'  Top-10 Weight      : {top10:.1f}%')

In [ ]:
if not holdings.empty and total_mv > 0:
    cum = weights_sorted.cumsum() * 100
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(range(1, len(cum) + 1), cum.values, marker='o', color=COLORS['highlight'])
    hline(ax, 50, label='50%', color=COLORS['neutral'])
    hline(ax, 80, label='80%', color=COLORS['neutral'])
    # Effective-N als vertikaler Marker
    if not np.isnan(effective_n):
        ax.axvline(effective_n, linestyle='--', color=COLORS['warn'], alpha=0.7,
                   label=f'Effective-N = {effective_n:.1f}')
    ax.set_xlabel('Anzahl der schwersten Positions')
    ax.set_ylabel('Kumuliertes Portfolio-Gewicht (%)')
    ax.set_title('Lorenz-artige Konzentrations-Kurve')
    ax.legend(); plt.tight_layout(); plt.show()

## 3. Allokations-Breakdown

In [ ]:
if not holdings.empty and total_mv > 0:
    def breakdown(col: str, label: str):
        b = (pos.assign(**{col: pos[col].fillna(f'unknown_{col}')})
                .groupby(col)['mv_ptf'].agg(['sum', 'count'])
                .rename(columns={'sum': 'mv_ptf_sum', 'count': 'n_positions'})
                .sort_values('mv_ptf_sum', ascending=False))
        b['pct'] = 100.0 * b['mv_ptf_sum'] / total_mv
        b.index.name = label
        return b

    for col, label in [('sector', 'Sector'),
                        ('industry', 'Industry'),
                        ('country', 'Country'),
                        ('instr_currency', 'Currency'),
                        ('asset_type', 'Asset Type')]:
        if col in pos.columns:
            tab = breakdown(col, label)
            print(f'== {label} ==')
            print(tab.to_string())
            print()

In [ ]:
if not holdings.empty and total_mv > 0:
    # Subplot grid: sector + currency + country + asset_type
    breakdowns = [
        ('sector',         'Sector'),
        ('instr_currency', 'Currency'),
        ('country',        'Country'),
        ('asset_type',     'Asset Type'),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, (col, label) in zip(axes.flat, breakdowns):
        if col not in pos.columns:
            ax.axis('off'); continue
        b = (pos.assign(**{col: pos[col].fillna(f'unknown')})
                .groupby(col)['mv_ptf'].sum()
                .sort_values(ascending=True))
        if b.empty or b.sum() == 0:
            ax.text(0.5, 0.5, f'no {label} data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(label); continue
        pct = 100.0 * b / b.sum()
        ax.barh(pct.index, pct.values, color=COLORS['highlight'])
        for i, v in enumerate(pct.values):
            ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)
        ax.set_xlabel('% of MV')
        ax.set_title(f'{label} ({len(b)} Buckets)')
    plt.tight_layout(); plt.show()

## 4. Korrelations-Matrix (90d Daily-Returns)

Werte ≥ 0.7 zwischen zwei Holdings sind ein Diversifikations-Warnsignal: in einem Drawdown bewegen sie sich gleichzeitig. ETFs zu allgemeinen Indizes haben naturgemäß hohe Korrelation untereinander.

In [ ]:
if not holdings.empty and total_mv > 0:
    since = date.today() - timedelta(days=RETURN_WINDOW_DAYS + 5)
    ids = pos['ref_instrument_id'].tolist()
    placeholders = ','.join(['?']*len(ids))
    with session() as con:
        q = df(con, f'''
            WITH ranked AS (
                SELECT ref_instrument_id, ts, close, source,
                       ROW_NUMBER() OVER (
                           PARTITION BY ref_instrument_id, ts
                           ORDER BY CASE source WHEN 'ib' THEN 1 WHEN 'yfinance' THEN 2 ELSE 9 END
                       ) AS rk
                FROM mkt_quotes_daily
                WHERE ts >= ? AND ref_instrument_id IN ({placeholders})
            )
            SELECT ref_instrument_id, ts, close FROM ranked WHERE rk = 1
            ORDER BY ref_instrument_id, ts
        ''', [since, *ids])

    if q.empty:
        print('Keine Quote-History — Korrelation übersprungen.')
        corr_matrix = pd.DataFrame()
    else:
        # Symbol-Map für lesbarere Labels
        sym_map = pos.set_index('ref_instrument_id')['symbol'].to_dict()
        q['ts'] = pd.to_datetime(q['ts'])
        wide = q.pivot(index='ts', columns='ref_instrument_id', values='close').sort_index()
        rets = wide.pct_change().dropna(how='all')
        # Nur Spalten mit genug Observations
        valid_cols = [c for c in rets.columns if rets[c].notna().sum() >= MIN_OBS_FOR_CORR]
        if len(valid_cols) < 2:
            print(f'Zu wenig Positions mit ≥ {MIN_OBS_FOR_CORR} Return-Observations.')
            corr_matrix = pd.DataFrame()
        else:
            corr_matrix = rets[valid_cols].corr()
            corr_matrix = corr_matrix.rename(columns=sym_map, index=sym_map)
            # Median pairwise (off-diagonal)
            tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            median_corr = tri.stack().median()
            print(f'Korrelations-Matrix: {len(valid_cols)} Positions, '
                  f'{int(rets[valid_cols].notna().sum().mean())} Mean-Observations')
            print(f'Median Pairwise Korrelation: {median_corr:.3f}')
            corr_matrix.round(2)

In [ ]:
if not holdings.empty and total_mv > 0 and 'corr_matrix' in dir() and not corr_matrix.empty:
    fig, ax = plt.subplots(figsize=(max(7, len(corr_matrix) * 0.6),
                                      max(6, len(corr_matrix) * 0.55)))
    im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr_matrix)))
    ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr_matrix)))
    ax.set_yticklabels(corr_matrix.index)
    # Annotate Werte
    for i in range(len(corr_matrix)):
        for j in range(len(corr_matrix)):
            v = corr_matrix.iat[i, j]
            color = 'white' if abs(v) > 0.5 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color=color)
    plt.colorbar(im, ax=ax, shrink=0.7).set_label('Korrelation')
    ax.set_title(f'Daily-Return-Korrelation, letzte {RETURN_WINDOW_DAYS}d (lokale Returns, keine FX-Adj.)')
    plt.tight_layout(); plt.show()

## 5. Risk-Decomposition (Marginal Contribution to Risk)

Wer trägt wieviel Portfolio-Vola? Formel:
$$\text{MCR}_i = \frac{w_i \cdot (\Sigma w)_i}{\sigma_p}, \quad \text{Share}_i = \frac{\text{MCR}_i \cdot w_i}{\sigma_p}$$

Wobei Σ die Kovarianz-Matrix der Daily-Returns ist und σ_p die Portfolio-Vola. Wenn ein Symbol 15% Weight aber 40% Risk-Share hat, ist es der Vola-Treiber.

In [ ]:
if not holdings.empty and total_mv > 0 and 'rets' in dir() and not rets.empty and len(valid_cols) >= 2:
    w_map = pos.set_index('ref_instrument_id')['weight'].to_dict()
    # Weights und Returns auf gemeinsame Reihenfolge ausrichten
    cols = valid_cols
    w = np.array([w_map.get(c, 0.0) for c in cols])
    # Re-normalize falls einige refs in rets aber nicht in w (sollte nicht passieren)
    if w.sum() > 0:
        w = w / w.sum()
    cov = rets[cols].cov().values
    # Daily-Vola -> annualisiert mit sqrt(252)
    sigma_p_daily = float(np.sqrt(w @ cov @ w))
    sigma_p_ann   = sigma_p_daily * np.sqrt(252)
    # MCR per position (daily); Share == w_i * MCR_i / sigma_p
    mcr = (cov @ w) / sigma_p_daily if sigma_p_daily > 0 else np.zeros_like(w)
    contrib = w * mcr
    share = contrib / sigma_p_daily if sigma_p_daily > 0 else contrib

    risk_df = pd.DataFrame({
        'symbol':           [sym_map.get(c, c) for c in cols],
        'weight_pct':       w * 100,
        'daily_vol_pct':    np.sqrt(np.diag(cov)) * 100,
        'risk_share_pct':   share * 100,
    }).sort_values('risk_share_pct', ascending=False)
    risk_df['weight_vs_risk'] = risk_df['risk_share_pct'] - risk_df['weight_pct']

    print(f'Portfolio Daily-Vola: {sigma_p_daily*100:.3f}%   |   Annualisiert: {sigma_p_ann*100:.2f}%')
    print()
    print('Risk-Decomposition (Pos sortiert nach Risk-Share):')
    risk_df.head(20)

In [ ]:
if not holdings.empty and total_mv > 0 and 'risk_df' in dir() and not risk_df.empty:
    plot_df = risk_df.head(15).sort_values('risk_share_pct', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(plot_df))))
    y = np.arange(len(plot_df))
    ax.barh(y - 0.2, plot_df['weight_pct'],     height=0.4,
            label='Weight',     color=COLORS['neutral'])
    ax.barh(y + 0.2, plot_df['risk_share_pct'], height=0.4,
            label='Risk-Share', color=COLORS['danger'])
    ax.set_yticks(y); ax.set_yticklabels(plot_df['symbol'])
    ax.set_xlabel('% des Portfolios')
    ax.set_title('Weight vs. Risk-Share — wo Risk > Weight, ist der Vola-Treiber')
    ax.legend(); plt.tight_layout(); plt.show()

## 6. Diagnostics (auto-erkannte Warnungen)

Ein paar Heuristiken die Du mit den Param-Cell-Schwellen tunen kannst. Wenn die Liste leer bleibt, ist das Portfolio nach diesen Regeln 'in Ordnung' — die Regeln sind aber bewusst grob.

In [ ]:
if not holdings.empty and total_mv > 0:
    warnings_list: list[str] = []

    # Position-Konzentration
    fat = pos[pos['weight'] * 100 > CONCENTRATION_WARN_PCT]
    for _, r in fat.iterrows():
        warnings_list.append(
            f'POSITION-KONZENTRATION: {r["symbol"]} = {r["weight"]*100:.1f}% '
            f'(Schwellwert {CONCENTRATION_WARN_PCT}%)'
        )

    # Sector-Konzentration
    if 'sector' in pos.columns:
        sb = (pos.assign(sector=pos['sector'].fillna('unknown'))
                  .groupby('sector')['mv_ptf'].sum())
        sb_pct = 100.0 * sb / total_mv
        for sec, pct in sb_pct.items():
            if pct > SECTOR_WARN_PCT:
                warnings_list.append(
                    f'SECTOR-KONZENTRATION: "{sec}" = {pct:.1f}% (Schwellwert {SECTOR_WARN_PCT}%)'
                )

    # Median-Korrelation
    if 'median_corr' in dir() and not np.isnan(median_corr) and median_corr > CORR_WARN:
        warnings_list.append(
            f'KORRELATION: Median pairwise = {median_corr:.2f} > {CORR_WARN} '
            f'(Holdings bewegen sich stark gemeinsam)'
        )

    # Risk-Decomposition: Position mit Risk-Share > 2x Weight
    if 'risk_df' in dir() and not risk_df.empty:
        for _, r in risk_df.iterrows():
            if r['weight_pct'] > 1 and r['risk_share_pct'] > 2 * r['weight_pct']:
                warnings_list.append(
                    f'VOLA-TREIBER: {r["symbol"]} weight={r["weight_pct"]:.1f}% '
                    f'risk-share={r["risk_share_pct"]:.1f}% '
                    f'(Risk/Weight = {r["risk_share_pct"]/r["weight_pct"]:.1f}x)'
                )

    # Currency-Konzentration: alles in einer währung?
    if 'instr_currency' in pos.columns:
        ccyb = (pos.groupby('instr_currency')['mv_ptf'].sum() / total_mv * 100)
        for ccy, pct in ccyb.items():
            if pct > 80 and ccy != PORTFOLIO_CCY:
                warnings_list.append(
                    f'WÄHRUNGS-EXPOSURE: {pct:.1f}% in {ccy} '
                    f'(starkes FX-Risiko vs {PORTFOLIO_CCY})'
                )

    if not warnings_list:
        print('Keine Warnungen anhand der konfigurierten Schwellen.')
        print('(Heißt nicht: alles ist gut. Heißt: keine Regel hat ausgelöst.)')
    else:
        print(f'{len(warnings_list)} Warnungen:')
        for w in warnings_list:
            print(f'  • {w}')